# FictiPay Datathon — Exploratory Data Analysis & System Audit

This notebook provides a thorough exploratory analysis of the **FictiPay** mobile wallet dataset. 
It serves as a diagnostic audit to understand the tables, columns, missing values, date ranges, and class distributions, while addressing critical findings like missing files and matching the competition checklist to this specific schema.

## Observation & Prediction Windows
- **Observation Window**: `2024-01-01` to `2024-03-31` (90 days). All features must be engineered exclusively from this window.
- **Prediction Window**: `2024-04-01` to `2024-04-30` (30 days). Churn is defined by the absence of transactions in this period.
- **Target Variable**: `CHURN = 1` if the customer made zero transactions in April 2024; `CHURN = 0` otherwise.

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define base paths
base_path = "./bkash-presents-nsucec-datathon/public"
print("Base path exists:", os.path.exists(base_path))
print("Files in public directory:", os.listdir(base_path))

: 

## 1. Train Labels & Churn Distribution

We inspect `train_labels.csv` to look at the total number of accounts and analyze the class imbalance.

In [ ]:
train_labels = pd.read_csv(os.path.join(base_path, "train_labels.csv"))
test_ids = pd.read_csv(os.path.join(base_path, "test.csv"))

print(f"Train Label Rows: {len(train_labels):,}")
print(f"Test ID Rows: {len(test_ids):,}")

churn_counts = train_labels["CHURN"].value_counts()
churn_rates = train_labels["CHURN"].value_counts(normalize=True)

print("\nChurn Counts:")
print(churn_counts)
print("\nChurn Rates (Imbalance):")
print(churn_rates)

# Plotting Class Imbalance
plt.figure(figsize=(6, 4))
plt.bar(['Active (0)', 'Churn (1)'], churn_counts, color=['#3b82f6', '#ef4444'], width=0.5)
plt.title(f"Churn Class Distribution (Active: {churn_rates[0]*100:.1f}%, Churned: {churn_rates[1]*100:.1f}%)")
plt.ylabel("Number of Customers")
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

## 2. Account Metadata (KYC)

The `kyc.parquet` contains account metadata. Only `Customer` accounts appear in the train/test sets; `Merchant` and `Biller` accounts appear as transaction endpoints.

In [ ]:
kyc = pd.read_parquet(os.path.join(base_path, "kyc.parquet"))
print(f"KYC Dimensions: {kyc.shape[0]:,} rows, {kyc.shape[1]} columns\n")
print("Columns:", list(kyc.columns))
print("\nMissing values per column:")
print(kyc.isnull().sum())

print("\nAccount Type distribution:")
print(kyc["ACCOUNT_TYPE"].value_counts())

print("\nGender distribution (including missing values):")
print(kyc["GENDER"].value_counts(dropna=False))

print("\nRegion distribution:")
print(kyc["REGION"].value_counts())

## 3. Transactions Audit & The Missing March File

The `transactions` folder contains monthly transaction records. 
**Key Finding**: In our check, there are only two files: `trx_2024-01.parquet` and `trx_2024-02.parquet`. **March transactions are missing**.
Let's load a sample and inspect the transaction types, source, and destination accounts.

In [ ]:
trx_files = sorted(glob.glob(os.path.join(base_path, "transactions", "*.parquet")))
print("Transaction files found:", [os.path.basename(f) for f in trx_files])

# Load a 100,000-row sample from January transactions
sample_trx = pd.read_parquet(trx_files[0]).head(100000)
print(f"\nSample transactions loaded from {os.path.basename(trx_files[0])}:")
print(sample_trx.head(5))
print("\nTransaction columns & types:")
print(sample_trx.dtypes)

# Check unique transaction types
print("\nTransaction Type Counts:")
print(sample_trx["TRX_TYPE"].value_counts())

# Map source and destination account types using KYC data
acct_map = kyc.set_index("ACCOUNT_ID")["ACCOUNT_TYPE"].to_dict()
sample_trx["SRC_TYPE"] = sample_trx["SRC_ACCOUNT"].map(acct_map)
sample_trx["DST_TYPE"] = sample_trx["DST_ACCOUNT"].map(acct_map)

print("\nSource vs Destination Account Type Crosstab:")
print(pd.crosstab(sample_trx["SRC_TYPE"].fillna("Unknown"), sample_trx["DST_TYPE"].fillna("Unknown")))

## 4. Day-End Balances & March Coverage

The `dayend_balance` folder contains the daily available balances for each account.
Unlike transactions, daily balances exist for **all three months (January, February, and March)**. 
We will use March balance trends (such as standard deviation, mean, balance drop from start to end) to represent active state during the month of March.

In [ ]:
bal_files = sorted(glob.glob(os.path.join(base_path, "dayend_balance", "*.parquet")))
print("Balance files found:", [os.path.basename(f) for f in bal_files])

# Load balance records for a single customer to trace their wallet balance over the 90 days
target_customer = "CUST00000001"
cust_bals = []
for f in bal_files:
    df = pd.read_parquet(f)
    cust_df = df[df["ACCOUNT_ID"] == target_customer].copy()
    cust_bals.append(cust_df)

cust_bal_df = pd.concat(cust_bals).sort_values("DATE")
print(f"\nSample balance trend for customer {target_customer}:")
print(cust_bal_df.head(5))

# Plot balance trend over time
plt.figure(figsize=(12, 5))
plt.plot(pd.to_datetime(cust_bal_df["DATE"]), cust_bal_df["AVAILABLE_BALANCE"], marker='o', color='#10b981', linewidth=1.5, markersize=3)
plt.title(f"90-Day Daily Available Balance Trend for Customer {target_customer}")
plt.xlabel("Date")
plt.ylabel("Available Balance (TK)")
plt.grid(True, linestyle='--', alpha=0.5)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 5. Reconciling the Checklist Mismatch

The competition checklist provided in `The Comprehensive Checklist for Building Robust, Explainable Customer Churn Prediction Systems.pdf` is built for the **IBM Telco Churn** dataset. 
To win the FictiPay Datathon, we map the Telco features to their FictiPay schema equivalents:

| Telco Concept (From Checklist) | FictiPay Equivalent Implementation | Business Hypothesis |
| :--- | :--- | :--- |
| **Tenure** | `days_since_open` = (2024-03-31 - `ACCOUNT_OPEN_DATE`) | Longer relationships have higher switching costs. |
| **Total Charges** | `total_trx_amt_jan_feb` | Total monetary throughput reflect active utilization. |
| **Monthly Charges** | `avg_monthly_spend` = `total_trx_amt_jan_feb` / 2 | Average spend per month reflects wallet utility. |
| **HasMultipleServices** | `trx_type_diversity` = Count of unique transaction types used (out of 5) | Users with multiple payment types are highly integrated. |
| **Contract Type** | `balance_stability` = CV of balance (`std` / `mean`) | Low-volatility daily balances reflect stable, recurring usage. |
| **PaymentMethod** | `primary_trx_type` = Mode of TRX_TYPE for the customer | Different user personas (P2P users vs. merchant payers) churn differently. |
| **Geospatial Features** | Target or one-hot encoded `REGION` | Regional service patterns and agent densities. |

## 6. Verification of train/test overlaps

We verify that all accounts in the train and test lists are Customer accounts and that there are no overlapping identifiers.

In [ ]:
train_set = set(train_labels["ACCOUNT_ID"])
test_set = set(test_ids["ACCOUNT_ID"])

overlap = train_set.intersection(test_set)
print(f"Overlap between train and test sets: {len(overlap)}")

# Check types in KYC
train_types = kyc[kyc["ACCOUNT_ID"].isin(train_set)]["ACCOUNT_TYPE"].unique()
test_types = kyc[kyc["ACCOUNT_ID"].isin(test_set)]["ACCOUNT_TYPE"].unique()
print(f"Account types in training labels: {train_types}")
print(f"Account types in test IDs: {test_types}")